# Step 5 — Inference across years + change detection

The payoff. Run the trained U-Net on cloud-masked composites for **2019, 2021, 2024**, predict land cover for each year, and **quantify the change** — per-class area over time and a forest-loss map.

**Two parts:**
- **Part A** exports a 2019/2021/2024 composite (6 bands + a valid mask) to Drive, one GeoTIFF per year. *(Run on a GEE-connected kernel; ~a few min each.)*
- **Part B** loads `unet_best.pt`, predicts each year over the full AOI, and produces the area time-series + change maps. *(Run on Colab with the GPU.)*

> We predict **all three years with the model** (not WorldCover for 2021) so the years are directly comparable. 'forest' = tree cover (incl. mature plantation), so we read deforestation as tree-cover **loss** between years.

---
# Part A — Export the yearly composites

In [ ]:
!pip install -q earthengine-api
import ee
ee.Authenticate()
ee.Initialize(project="landcover-riau")

In [ ]:
AOI_BBOX = [102.9, -0.6, 103.4, -0.1]
region = ee.Geometry.Rectangle(AOI_BBOX)
S2_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]
EXPORT_CRS = "EPSG:32648"
DRIVE_FOLDER = "landcover_riau"
YEARS = [2019, 2021, 2024]


def s2_composite(year, reg, window=("06-01", "09-30"), threshold=0.6):
    start, end = f"{year}-{window[0]}", f"{year}-{window[1]}"
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(reg).filterDate(start, end))
    cs = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
    s2 = (s2.linkCollection(cs, ["cs_cdf"])
            .map(lambda i: i.updateMask(i.select("cs_cdf").gte(threshold))))
    return s2.median().clip(reg)


def export_year(year):
    img = s2_composite(year, region).select(S2_BANDS)
    valid = img.mask().reduce(ee.Reducer.min()).rename("valid")   # 1 where all bands present
    stack = img.addBands(valid).unmask(0).toFloat()               # 7 bands: 6 img + valid
    task = ee.batch.Export.image.toDrive(
        image=stack, description=f"pred_{year}", folder=DRIVE_FOLDER,
        fileNamePrefix=f"pred_{year}", region=region, scale=10,
        crs=EXPORT_CRS, maxPixels=1e10,
    )
    task.start()
    return task


tasks = {y: export_year(y) for y in YEARS}
print("started:", {y: t.id for y, t in tasks.items()})

In [ ]:
# Optional: wait for all three exports to finish.
import time
while any(t.active() for t in tasks.values()):
    print({y: t.status()["state"] for y, t in tasks.items()}); time.sleep(30)
print("final:", {y: t.status()["state"] for y, t in tasks.items()})

---
# Part B — Predict each year + quantify change

Run on Colab with the **GPU** on, once all three `pred_<year>.tif` files are in Drive.

In [ ]:
!pip install -q segmentation-models-pytorch rasterio
import numpy as np, rasterio, torch, pandas as pd
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from google.colab import drive
drive.mount("/content/drive")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
DRIVE = "/content/drive/MyDrive/landcover_riau"
CKPT  = f"{DRIVE}/unet_best.pt"
YEARS = [2019, 2021, 2024]
N_CLASSES, N_BANDS = 4, 6
REFL_SCALE = 10000.0
TILE = 256
CLASSES = {0: "forest", 1: "agriculture", 2: "urban", 3: "water"}
CLASS_COLORS = ["#1b7837", "#e6c200", "#d7191c", "#2c7fb8"]
PIXEL_KM2 = (10 * 10) / 1e6        # 0.0001 km^2 per 10 m pixel

model = smp.Unet("resnet34", encoder_weights=None, in_channels=N_BANDS, classes=N_CLASSES).to(device)
model.load_state_dict(torch.load(CKPT, map_location=device))
model.eval()
print("loaded model from", CKPT)

## Predict the whole AOI

The AOI is far too big to feed the U-Net at once, so we pad to a multiple of 256, predict in batched 256×256 patches, and stitch the result back into one full-size label map.

In [ ]:
def predict_full(img, tile=TILE, bs=16):
    """img: (6,H,W) normalised. Returns (H,W) uint8 class map."""
    C, H, W = img.shape
    ph, pw = (-H) % tile, (-W) % tile
    imgp = np.pad(img, ((0, 0), (0, ph), (0, pw)))
    Hp, Wp = imgp.shape[1:]
    out = np.zeros((Hp, Wp), np.uint8)
    coords = [(y, x) for y in range(0, Hp, tile) for x in range(0, Wp, tile)]
    with torch.no_grad():
        for k in range(0, len(coords), bs):
            chunk = coords[k:k + bs]
            batch = torch.stack([torch.from_numpy(imgp[:, y:y + tile, x:x + tile]) for y, x in chunk]).to(device)
            pred = model(batch).argmax(1).cpu().numpy().astype(np.uint8)
            for (y, x), p in zip(chunk, pred):
                out[y:y + tile, x:x + tile] = p
    return out[:H, :W]


preds, valids = {}, {}
for yr in YEARS:
    with rasterio.open(f"{DRIVE}/pred_{yr}.tif") as src:
        arr = np.nan_to_num(src.read(), nan=0.0)        # (7,H,W): 6 img + valid
    img = arr[:6].astype(np.float32) / REFL_SCALE
    preds[yr] = predict_full(img)
    valids[yr] = arr[6] > 0.5                            # True where imagery is real
    print(f"{yr}: predicted {preds[yr].shape}, valid {100*valids[yr].mean():.1f}%")

## Per-class area over time

Percentages are of the **valid** (non-cloud-hole) area each year, so missing data doesn't skew the totals.

In [ ]:
rows = {}
for yr in YEARS:
    p, v = preds[yr], valids[yr]
    tot = v.sum()
    rows[yr] = {CLASSES[c]: 100 * ((p == c) & v).sum() / tot for c in range(N_CLASSES)}
mix = pd.DataFrame(rows).T.round(1)

forest_km2 = {yr: ((preds[yr] == 0) & valids[yr]).sum() * PIXEL_KM2 for yr in YEARS}
mix["forest_km2"] = [round(forest_km2[yr], 1) for yr in YEARS]
print(mix)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(YEARS, [mix.loc[y, "forest"] for y in YEARS], "o-", color="#1b7837")
ax[0].set_title("Forest cover over time"); ax[0].set_ylabel("% of valid area"); ax[0].set_xticks(YEARS)
for c in range(N_CLASSES):
    ax[1].plot(YEARS, [mix.loc[y, CLASSES[c]] for y in YEARS], "o-", color=CLASS_COLORS[c], label=CLASSES[c])
ax[1].set_title("All classes"); ax[1].set_ylabel("% of valid area"); ax[1].set_xticks(YEARS); ax[1].legend()
plt.tight_layout(); plt.show()

## Land-cover maps + forest-loss map

Predicted land cover for each year, then a change map for **2019 → 2024**: red = forest lost (was tree cover, now isn't), blue = forest gained (regrowth/replant). Only pixels valid in *both* years are compared.

In [ ]:
cmap = ListedColormap(CLASS_COLORS)
fig, axes = plt.subplots(1, len(YEARS), figsize=(5 * len(YEARS), 5))
for ax, yr in zip(axes, YEARS):
    shown = np.where(valids[yr], preds[yr], np.nan)
    ax.imshow(shown, cmap=cmap, vmin=0, vmax=N_CLASSES - 1)
    ax.set_title(f"land cover {yr}"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
both = valids[2019] & valids[2024]
loss = (preds[2019] == 0) & (preds[2024] != 0) & both     # forest -> non-forest
gain = (preds[2019] != 0) & (preds[2024] == 0) & both     # non-forest -> forest

change = np.zeros(loss.shape, np.uint8)
change[gain] = 1
change[loss] = 2
change_cmap = ListedColormap(["#eeeeee", "#2c7fb8", "#d7191c"])  # stable / gain / loss

plt.figure(figsize=(7, 7))
plt.imshow(change, cmap=change_cmap, vmin=0, vmax=2)
plt.title("Forest change 2019 -> 2024  (red = loss, blue = gain)"); plt.axis("off"); plt.show()

print(f"forest LOST 2019->2024: {loss.sum() * PIXEL_KM2:6.1f} km^2")
print(f"forest GAINED         : {gain.sum() * PIXEL_KM2:6.1f} km^2")
print(f"net change            : {(gain.sum() - loss.sum()) * PIXEL_KM2:6.1f} km^2")

## Read-out

What to look for and note for the write-up:

- **Direction + magnitude** of the forest trend (the time-series and net km²). A decline is the expected deforestation signal.
- **Where** loss concentrates on the change map — edges of existing clearings / plantation fronts.
- **Caveats to state honestly:** only 3 snapshots; pixel-level change is noisy (some 'change' is just model disagreement year-to-year, not real conversion); and 'forest' includes plantation, so this is tree-cover change, not natural-forest-only.

**Next (Step 6):** bring in **World Bank palm-oil prices** and **Hansen annual forest loss (2001–2023)** for the lagged price→loss regression — the long, statistically meaningful economic story that complements these 3 high-detail snapshots.